%% [markdown]
# GPN-Star backprop parity — CPU / CUDA / Trainium  (DEV mode)
One backward through the vendored **full model** (`GPNStarForMaskedLM`) on every device; checks that
**all of the model's own parameter gradients** match CPU — i.e. the whole editable definition trains
consistently on Trainium. `cuda` auto-skips when absent.
`NEURON_RT_VISIBLE_CORES=0 jupyter nbconvert --to notebook --execute 02_backprop_parity.ipynb`.

⚠️ Gradient parity is magnitude-aware, not raw cosine: near-zero grads are fp noise. A tensor passes if
`cosine >= 0.99` OR its max-abs diff is `<= 1e-3` of the global grad scale.

In [1]:
# %%
import os
os.environ.setdefault("NEURON_RT_VISIBLE_CORES", "0")
import sys
sys.path.insert(0, os.path.abspath("../src"))
import torch
import torch.nn.functional as F
import gpn_star_reference as R

[W709 03:29:50.967372809 OperatorEntry.cpp:208] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::gather.out(Tensor self, int dim, Tensor index, *, bool sparse_grad=False, Tensor(a!) out) -> Tensor(a!)
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: PrivateUse1
  previous kernel: registered at /pytorch/build/aten/src/ATen/RegisterCPU_3.cpp:7637
       new kernel: registered at NeuronDispatcher.cpp:0 (function operator())
/home/user/miniconda3/envs/gpn-vep/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def devices():
    devs = ["cpu"]
    if torch.cuda.is_available():
        devs.append("cuda")
    try:
        import torch_neuronx  # noqa: F401
        devs.append("neuron")
    except Exception as e:
        print("neuron unavailable:", e)
    return devs

In [3]:
DEVICES = devices()
print("torch", torch.__version__, "| devices:", DEVICES)

torch 2.11.0+cpu | devices: ['cpu', 'neuron']


In [4]:
# %% [markdown]
# ## One backward per device through the full model; collect all param grads
# %%
def run_backprop(device):
    model = R.load(device)
    for p in model.parameters():
        p.requires_grad_(True)
    model.zero_grad(set_to_none=True)
    inputs = R.build_inputs()
    out = model(*[t.to(device) for t in inputs])
    R.loss_fn(out).backward()
    if device == "neuron":
        import torch_neuronx; torch_neuronx.synchronize()
    return {n: p.grad.detach().float().cpu() for n, p in model.named_parameters() if p.grad is not None}

In [5]:
results = {d: run_backprop(d) for d in DEVICES}
for d in DEVICES:
    print(f"{d:7s} grad_tensors={len(results[d])}")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 124583.29it/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 131988.59it/s]


/home/user/miniconda3/envs/gpn-vep/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 142987.64it/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 134816.91it/s]

cpu     grad_tensors=240
neuron  grad_tensors=240


In [6]:
# %% [markdown]
# ## Magnitude-aware gradient comparison vs CPU
# %%
def compare(ref, test):
    scale = max(g.abs().max().item() for g in ref.values()) or 1.0
    real = []
    for n, gr in ref.items():
        a, b = gr.flatten(), test[n].flatten()
        if F.cosine_similarity(a, b, dim=0).item() < 0.99 and (a - b).abs().max().item() / scale > 1e-3:
            real.append((n, (a - b).abs().max().item()))
    return scale, real

In [7]:
ref, all_ok = results["cpu"], True
for d in DEVICES:
    if d == "cpu":
        continue
    scale, real = compare(ref, results[d])
    print(f"\n{d} vs cpu: {len(ref)} grad tensors | global |grad|={scale:.3e}")
    print(f"  matched: {len(ref) - len(real)}/{len(ref)}")
    for n, ad in real:
        print(f"  REAL DIFF {n}  absdiff={ad:.3e}")
    all_ok = all_ok and not real


neuron vs cpu: 240 grad tensors | global |grad|=6.102e+02
  matched: 240/240


In [8]:
print("\nBACKPROP PARITY:", "PASS" if all_ok else "FAIL")
assert all_ok, "gradients diverged across devices (beyond near-zero fp noise)"


BACKPROP PARITY: PASS
